# CodeAlpha Superstore Exploratory Data Analysis

## 1. Introduction & Project Overview

**Author:** Lalit
**Project:** CodeAlpha Data Analytics Internship - Superstore EDA
**Version:** 1.1
**Purpose:** This project performs a comprehensive, production-grade Exploratory Data Analysis (EDA) on the Superstore dataset. The objective is to uncover actionable business intelligence through rigorous statistical and visual analysis.

---
### Table of Contents
- [1. Introduction & Project Overview](#1.-Introduction-&-Project-Overview)
- [2. Business Problem Definition](#2.-Business-Problem-Definition)
- [3. Dataset Context & Dictionary](#3.-Dataset-Context-&-Dictionary)
- [4. Data Loading & Overview](#4.-Data-Loading-&-Overview)
- [5. Data Cleaning](#5.-Data-Cleaning)
- [6. Feature Engineering](#6.-Feature-Engineering)
- [7. Univariate Analysis](#7.-Univariate-Analysis)
- [8. Bivariate Analysis](#8.-Bivariate-Analysis)
- [9. Multivariate Analysis](#9.-Multivariate-Analysis)
- [10. Time Series Analysis](#10.-Time-Series-Analysis)
- [11. Correlation Analysis](#11.-Correlation-Analysis)
- [12. Outlier Detection](#12.-Outlier-Detection)
- [13. Business Insights](#13.-Business-Insights)
- [14. Recommendations](#14.-Recommendations)
- [15. Conclusion](#15.-Conclusion)

## 2. Business Problem Definition
The objective is to identify key trends, patterns, and insights from the sales data to help the business make data-driven decisions. By understanding which regions, segments, and product categories perform best (or worst), leadership can optimize inventory, tailor marketing strategies, and improve overall operational efficiency.

## 3. Dataset Context & Dictionary
The dataset contains transactional records of a US-based superstore. 

### Data Dictionary
| Feature | Description | Data Type Expectation |
|---------|-------------|-----------------------|
| **Row ID** | Unique identifier for each row | Integer |
| **Order ID** | Unique identifier for the order | Categorical/String |
| **Order Date** | Date the order was placed | Datetime |
| **Ship Date** | Date the order was shipped | Datetime |
| **Ship Mode** | Selected shipping method | Categorical |
| **Customer ID** | Unique identifier for the customer | Categorical/String |
| **Customer Name** | Full name of the customer | String |
| **Segment** | Customer segment (Consumer, Corporate, Home Office) | Categorical |
| **Country, City, State, Postal Code** | Geographic location details | Categorical/Geospatial |
| **Region** | Macro-region of the US | Categorical |
| **Product ID** | Unique identifier for the product | Categorical/String |
| **Category & Sub-Category** | Product categorization hierarchy | Categorical |
| **Product Name** | Full name of the product | String |
| **Sales** | Revenue generated from the transaction | Numerical (Float) |

## 4. Data Loading & Overview

In this section, we load the Superstore dataset into a pandas DataFrame. Loading the data is the first step in our analysis pipeline. We will also perform an initial inspection to understand the dataset's structure, size, and the types of information it contains. This high-level overview helps us identify potential data quality issues and informs our subsequent cleaning and feature engineering steps.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Set pandas display options for better visibility
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
warnings.filterwarnings('ignore')

# Load the dataset
df = pd.read_csv('train.csv')

### 4.1 Dataset Shape & Columns
**Business Context:** Understanding the volume of our data (number of transactions/rows and features/columns) gives us a sense of scale. Identifying the exact columns available allows us to formulate initial business hypotheses (e.g., segmenting sales by region or category).

In [ ]:
print(f'The dataset contains {df.shape[0]} rows (transactions) and {df.shape[1]} columns (features).\n')
print('Columns in the dataset:')
for col in df.columns:
    print(f'- {col}')

### 4.2 Data Inspection: head(), tail(), and sample()
**Business Context:** Visually inspecting a few rows of the data confirms that it loaded correctly and gives us a tangible feel for what a single transaction looks like. We look at the top (`head`), bottom (`tail`), and a random slice (`sample`) to ensure structural consistency throughout the file.

In [ ]:
# View the first 5 records
df.head()

In [ ]:
# View the last 5 records
df.tail()

In [ ]:
# View 5 random records to spot any inconsistencies in the middle of the dataset
df.sample(5, random_state=42)

### 4.3 Dataset Info & Data Types
**Business Context:** The `info()` method provides a concise summary of the DataFrame, including data types and the number of non-null values. This is critical for identifying missing data early on. Missing data can skew analysis and must be handled appropriately to ensure accurate business reporting.

In [ ]:
df.info()

### 4.3.1 Missing Value Percentage Table
**Business Context:** Quantifying the exact percentage of missing values helps us prioritize our data cleaning strategy. If a column is missing >50% of its data, we might drop it; if it's missing <5%, we might impute it.

In [ ]:
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'Missing Values': df.isnull().sum(), 'Percentage (%)': missing_pct})
missing_df[missing_df['Missing Values'] > 0].sort_values(by='Percentage (%)', ascending=False)

### 4.3.2 Memory Usage
**Business Context:** Checking memory usage ensures our dataset scales efficiently, especially when deploying into production environments or processing massive transaction logs.

In [ ]:
memory_mb = df.memory_usage(deep=True).sum() / 1024**2
print(f'Total Memory Usage: {memory_mb:.2f} MB')

### 4.4 Descriptive Statistics
**Business Context:** Descriptive statistics (`describe()`) provide a mathematical summary of our columns. 
- **Numerical:** Shows the average transaction value, spread, and range. This helps in immediately spotting potential outliers (e.g., massive single orders).
- **Categorical:** Shows unique values (cardinality), the most frequent category, and its frequency, helping us understand segment dominance.

In [ ]:
# Statistical summary for numerical columns (transposed for readability)
df.describe().T

In [ ]:
# Statistical summary for categorical columns (transposed for readability)
df.describe(include='object').T

### 4.5 Dataset Health Summary & Initial Observations

**Dataset Health Summary:**
- **Memory Footprint:** The dataset is extremely lightweight, confirming we can perform in-memory operations using Pandas without optimization.
- **Missing Data Severity:** Only `Postal Code` exhibits missing values (11 out of 9,800 or ~0.11%). This is a highly healthy dataset.

**Data Quality & Structure:**
1.  **Data Types:** `Order Date` and `Ship Date` are `object` (string) type. We must cast these to `datetime` to unlock time series analysis (e.g., seasonality, shipping delays). `Postal Code` is loaded as numeric but represents a geographical category.
2.  **Outliers:** The `Sales` column has a mean of ~$230.77 but a max of $22,638.48. The 75th percentile is only $209.50. This indicates a heavily right-skewed distribution, typical in retail, driven by a few massive corporate or bulk orders.

**Business Intelligence Opportunities:**
1.  **Geographic Performance:** With Country, Region, State, and City available, we can pinpoint exactly where our revenue is concentrated and where we might be underperforming.
2.  **Product Strategy:** We have 15 unique Sub-Categories and 3 main Categories. We can identify the "cash cows" versus low-performing product lines.
3.  **Customer Segmentation:** The dataset tracks 3 unique Segments and over 790 unique customers. Analyzing purchasing behavior across segments will inform targeted marketing strategies.
4.  **Operational Efficiency:** By comparing `Order Date` and `Ship Date` alongside `Ship Mode`, we can evaluate our logistics performance.

## 5. Data Cleaning

## 6. Feature Engineering

## 7. Univariate Analysis

## 8. Bivariate Analysis

## 9. Multivariate Analysis

## 10. Time Series Analysis

## 11. Correlation Analysis

## 12. Outlier Detection

## 13. Business Insights

## 14. Recommendations

## 15. Conclusion